# 第 6 章の実習　表の結合

那珂川市（福岡県）か丹波篠山市（兵庫県）を一つ選び，2015 年から 2020 年に人口がどう変わったかを報告します。同じ地域の二つの時点を，統計局の変更記録を根拠に対応させます。

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

最初のコードはデータが置いてあるフォルダーを探し，`code` を 5 桁の文字列として読みます。実行すると，2015 年，2020 年の順に行数が表示されます。

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

file_2015 = "census-municipalities-2015.csv"
file_2020 = "census-municipalities-2020.csv"
data_folder = Path("data")
if not (data_folder / file_2015).is_file():
    data_folder = Path("../data")
if not (data_folder / file_2015).is_file():
    data_folder = Path(".")

path_2015 = data_folder / file_2015
path_2020 = data_folder / file_2020
if not path_2015.is_file():
    path_2015 = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_2015
if not path_2020.is_file():
    path_2020 = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_2020

code_type = {"code": "str"}
census_2015 = pd.read_csv(path_2015, dtype=code_type)
census_2020 = pd.read_csv(path_2020, dtype=code_type)
len(census_2015), len(census_2020)

## 1. 相手を探す（約 5 分）

下のセルの `selected_code` に選んだ市の 2020 年のコードを入れます。那珂川市は `40231`，丹波篠山市は `28221` です。まず 2020 年の行を表示します。

In [ ]:
selected_code = "40231"
selected_2020 = census_2020.loc[census_2020["code"] == selected_code].copy()
selected_2020

選んだ市を下の表で探し，旧名称と旧コード，何が変わったかを読みます。この記録は，2015 年と 2020 年の国勢調査の間の変更をまとめたものです。[統計局の市区町村の変更情報・組替人口一覧](https://www.stat.go.jp/data/kokusei/2020/kekka/pdf/r04.pdf)

| 変更日 | 旧名称・コード | 内容 | 新名称・コード |
|---|---|---|---|
| 2016 年 10 月 10 日 | 富谷町 `04423` | 市制施行 | 富谷市 `04216` |
| 2018 年 10 月 1 日 | 那珂川町 `40305` | 市制施行 | 那珂川市 `40231` |
| 2019 年 5 月 1 日 | 篠山市 `28221` | 名称変更 | 丹波篠山市 `28221` |

**予想**　2020 年の `code` のまま 2015 年表を探すと相手が見つかりますか。変更表のどこからそう考えたか，一文で書いてください。

（ここに記入）

## 2. 二つの行を組にする（約 9 分）

富谷市の見本では，2020 年の `code` と，2015 年の相手を探す `comparison_code` が別々にあります。見本を実行し，出力ではどの列に旧コードが残るかを確かめます。

In [ ]:
tomiya_2020 = census_2020.loc[census_2020["code"] == "04216"].copy()
tomiya_2020["comparison_code"] = "04423"

tomiya = tomiya_2020.merge(
    census_2015,
    left_on="comparison_code",
    right_on="code",
    how="left",
    suffixes=("_2020", "_2015"),
    validate="one_to_one",
    indicator=True,
)
tomiya[["code_2020", "municipality_2020", "code_2015", "municipality_2015", "_merge"]]

自分が選んだ市の旧コードを，次のセルの `comparison_code` に文字列で入力します。2015 年表からそのコードの行を表示します。変更記録と，表示された都道府県・名称が対応するか確認します。

In [ ]:
comparison_code = ""

if comparison_code:
    display(census_2015.loc[census_2015["code"] == comparison_code])

次は二つの年を結合します。下のセルの `#` を外し，`left_on` と `right_on` の `""` に，照合に使う列名を入れます。最初のコメント行は，元の `code` も読み返せるよう，新しい列に比較用コードを保存します。

- 左の表 `selected_2020` は，元の `code` と，追加する `comparison_code` を持ちます。
- 右の表 `census_2015` では，2015 年のコードが `code` 列にあります。

自分が選んだ旧コードが左表のどの列に入るかを考え，右表のどの列からその値を探すか決めてください。

In [ ]:
selected_comparison = None

# selected_2020["comparison_code"] = comparison_code
# selected_comparison = selected_2020.merge(
#     census_2015,
#     left_on="",
#     right_on="",
#     how="left",
#     suffixes=("_2020", "_2015"),
#     validate="one_to_one",
#     indicator=True,
# )

次のセルで，結合した行と人口の差を表示します。`_merge` が `both` なら，指定したキーに対応する行が両方の表に見つかっています。選んだ相手が公式記録の旧名称・旧コードと合うか，人口が手前で表示した元の 2 行と合うかを確かめます。`left_only` のときは，右の `code` と照合する 5 桁を読み返します。

In [ ]:
if selected_comparison is None:
    print("結合を記入して実行すると，ここに結果を表示できます。")
else:
    selected_comparison["population_change"] = (
        selected_comparison["population_2020"] - selected_comparison["population_2015"]
    )
    display(selected_comparison[[
        "code_2020", "municipality_2020", "code_2015", "municipality_2015", "_merge",
    ]])
    display(selected_comparison[[
        "population_2015", "population_2020", "population_change",
    ]])

## 3. 人口の変化を報告する（約 6 分）

選んだ地域について，計算と資料をたどれるように 3〜4 文で書きます。

1. 地域名，二つの調査時点それぞれの人口と，何人増えたか・減ったか。
2. どの二つのコードを照合したか。変更記録の何を根拠に同じ地域の組としたか。
3. 人口が変わった理由を調べるため，次にどの情報を確かめるか。その情報から何を知りたいか。

（ここに記入）

提出前にカーネルを再起動し，全セルを上から実行します。1 で選んだ市について差が表示され，その市の報告が残ったノートブックを保存してください。

### もう一つの地域（任意）

もう一方でも，1 の地域選択からやり直せます。二つの地域で，名称が変わったこととコードが変わったことはどう違いますか。結果を保存するなら，一つ目の報告と区別して書きます。

### 出典

総務省統計局の平成 27 年・令和 2 年国勢調査から，市町村と東京都特別区の総人口を教材用に抽出・加工しました。各年 10 月 1 日現在の人数です。[2015 年の原表](https://www.e-stat.go.jp/stat-search/files?cycle=0&layout=datalist&month=0&tclass1=000001089055&tclass2=000001089056&toukei=00200521&tstat=000001080615&year=20150) ・ [2020 年の原表](https://www.e-stat.go.jp/stat-search/files?cycle=0&layout=datalist&page=1&stat_infid=000032142410&tclass1=000001136466&toukei=00200521&tstat=000001136464)